In [3]:
pip install bs4 requests

In [2]:
import json
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ==============================================================================
# KONFIGURASI DAN HEADER BROWSER
# ==============================================================================
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Daftar ID Sub-Genre Publik Netflix untuk memperbanyak pencarian URL
GENRE_IDS = [
    "83",     # TV Shows / Series
    "34399",  # Movies
    "1365",   # Action & Adventure
    "5763",   # Dramas
    "8711",   # Horror
    "6548",   # Comedies
    "8883",   # Romance
    "7424",   # Anime
    "6839",   # Documentaries
    "10118",  # Superhero
    "31574",  # Classics
    "5977",   # LGBTQ
    "783",    # Children & Family
    "8933",   # Thrillers
    "4370"    # Sports
]

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================
def get_span_text(soup, header_text):
    """Helper aman untuk mengambil teks span setelah tag header tanpa AttributeError"""
    h_tag = soup.find(['h4', 'h3', 'span'], string=header_text)
    if h_tag and h_tag.find_next_sibling('span'):
        return h_tag.find_next_sibling('span').text.strip()
    return "NA"

def collect_movie_urls(genre_list):
    """Mengumpulkan URL unik film/series dari belasan sub-genre Netflix."""
    movie_urls = set()
    print("=== STEP 1: Mengumpulkan URL dari Multi-Genre Netflix ===")

    for gid in genre_list:
        url = f"https://www.netflix.com/us/browse/genre/{gid}"
        print(f"Scraping daftar link dari Genre ID: {gid} ...")
        try:
            res = requests.get(url, headers=HEADERS, timeout=10)
            if res.status_code == 200:
                soup = BeautifulSoup(res.text, 'html.parser')
                for a in soup.find_all('a', href=True):
                    href = a['href']
                    if '/title/' in href:
                        clean_href = href.split('?')[0]
                        full_url = clean_href if clean_href.startswith('http') else f"https://www.netflix.com{clean_href}"
                        movie_urls.add(full_url)
            time.sleep(0.2)
        except Exception as e:
            print(f"Gagal mengambil dari Genre {gid}: {e}")

    print(f"\n[SELESAI STEP 1] Total URL film/series unik terkumpul: {len(movie_urls)}\n")
    return list(movie_urls)

def scrape_title_details(url):
    """Mengambil metadata detail tiap film/series dari halaman publik Netflix."""
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code != 200:
            return None, None

        soup = BeautifulSoup(res.text, 'html.parser')

        # 1. Ekstrak Film_ID dari URL
        film_id = url.split('/title/')[-1].split('?')[0] if '/title/' in url else "NA"

        # 2. Parse Microdata JSON-LD
        json_scripts = soup.find_all('script', type='application/ld+json')
        data = {}
        for script in json_scripts:
            if script.text and 'schema.org' in script.text:
                try:
                    data = json.loads(script.text)
                    break
                except Exception:
                    continue

        # 3. Ekstraksi Data Utama dari JSON-LD
        title_name = soup.h1.text.strip() if soup.h1 else data.get('name', 'Unknown Title')

        # Ekstraksi Genre Utama
        raw_genre = data.get('genre', 'NA')
        genre_utama = raw_genre[0] if isinstance(raw_genre, list) else raw_genre

        # Ekstraksi Tahun Rilis (Ambil 4 digit pertama)
        release_date = str(data.get('dateCreated', 'NA'))
        release_year = release_date[:4] if release_date != 'NA' else 'NA'

        rating = data.get('contentRating', 'NA')
        number_of_seasons = data.get('numberOfSeasons', 'NA')
        synopsis = data.get('description', 'NA')

        # Logika Content Type
        content_type = "TV Series" if number_of_seasons != 'NA' else "Movie"

        # 4. Ekstraksi Elemen HTML Tambahan
        audio = get_span_text(soup, 'Audio')
        subtitles = get_span_text(soup, 'Subtitles')
        cast = get_span_text(soup, 'Cast')

        # Cari Director atau Creator (Pencegahan variasi nama tag di Netflix)
        director = get_span_text(soup, 'Creators')
        if director == "NA":
            director = get_span_text(soup, 'Director')

        # Cari Moods (Pencegahan variasi kata 'show' / 'movie')
        moods = get_span_text(soup, 'This show is ...')
        if moods == "NA":
            moods = get_span_text(soup, 'This movie is ...')

        # 5. Logika Pemecahan 8 Kolom Related Genres
        related_genres_text = get_span_text(soup, 'Genres')
        rg_list = [g.strip() for g in related_genres_text.split(',')] if related_genres_text != 'NA' else []

        # Isi otomatis dengan "NA" jika jumlah sub-genre kurang dari 8
        while len(rg_list) < 8:
            rg_list.append("NA")

        # 6. Susun Dictionary Final
        details = {
            "Film_ID": film_id,
            "Title": title_name,
            "Content_Type": content_type,
            "Genre_Utama": genre_utama,
            "Related_Genre_1": rg_list[0],
            "Related_Genre_2": rg_list[1],
            "Related_Genre_3": rg_list[2],
            "Related_Genre_4": rg_list[3],
            "Related_Genre_5": rg_list[4],
            "Related_Genre_6": rg_list[5],
            "Related_Genre_7": rg_list[6],
            "Related_Genre_8": rg_list[7],
            "Age_Rating": rating,
            "Release_Year": release_year,
            "Seasons": number_of_seasons,
            "Synopsis": synopsis,
            "Cast": cast,
            "Director": director,
            "Moods": moods,
            "Audio": audio,
            "Subtitles": subtitles,
            "URL": url
        }

        return details, title_name

    except Exception as e:
        return None, None

# ==============================================================================
# MAIN PIPELINE
# ==============================================================================
def main():
    # Step 1: Kumpulkan URL
    all_urls = collect_movie_urls(GENRE_IDS)

    if not all_urls:
        print("Tidak ada URL yang berhasil terkumpul.")
        return

    print("=== STEP 2: Scraping Detail Setiap Film/Series ===")
    results_dict = {}
    results_list = []

    total = len(all_urls)

    # Catatan: Kalau mau tes cepat dulu misal 50 URL pertama, ganti 'all_urls' jadi 'all_urls[:50]'
    for index, url in enumerate(all_urls, 1):
        print(f"[{index}/{total}] Fetching: {url}")
        details, title = scrape_title_details(url)

        if details and title:
            results_dict[title] = details
            results_list.append(details)

    # Step 3: Simpan Hasil Ke File
    print("\n=== STEP 3: Menyimpan Hasil Scraping ===")

    # Simpan JSON
    with open("netflix.json", "w", encoding="utf-8") as f:
        json.dump(results_dict, f, indent=4, ensure_ascii=False)
    print("[SUKSES] Data tersimpan ke 'netflix.json'")

    # Simpan CSV
    df = pd.DataFrame(results_list)
    df.to_csv("netflix.csv", index=False, encoding="utf-8")
    print("[SUKSES] Data tersimpan ke 'netflix.csv'")
    print(f"\n[SELESAI] Total {len(df)} data film/series berhasil diekstrak dan siap dianalisis!")

if __name__ == "__main__":
    main()

=== STEP 1: Mengumpulkan URL dari Multi-Genre Netflix ===
Scraping daftar link dari Genre ID: 83 ...
Scraping daftar link dari Genre ID: 34399 ...
Scraping daftar link dari Genre ID: 1365 ...
Scraping daftar link dari Genre ID: 5763 ...
Scraping daftar link dari Genre ID: 8711 ...
Scraping daftar link dari Genre ID: 6548 ...
Scraping daftar link dari Genre ID: 8883 ...
Scraping daftar link dari Genre ID: 7424 ...
Scraping daftar link dari Genre ID: 6839 ...
Scraping daftar link dari Genre ID: 10118 ...
Scraping daftar link dari Genre ID: 31574 ...
Scraping daftar link dari Genre ID: 5977 ...
Scraping daftar link dari Genre ID: 783 ...
Scraping daftar link dari Genre ID: 8933 ...
Scraping daftar link dari Genre ID: 4370 ...

[SELESAI STEP 1] Total URL film/series unik terkumpul: 2638

=== STEP 2: Scraping Detail Setiap Film/Series ===
[1/2638] Fetching: https://www.netflix.com/us/title/81266458
[2/2638] Fetching: https://www.netflix.com/us/title/81754156
[3/2638] Fetching: https://www.n